# 07. 메모리(Memory)와 체크포인터(Checkpointer)

> LangGraph는 매 superstep마다 상태를 저장해 멀티턴 대화와 실패 복구를 가능하게 해요. `MemorySaver` + `thread_id` 만으로 어떻게 영속적인 챗봇이 만들어지는지 봐요.

LangGraph에서 그래프는 기본적으로 **상태 비보존(stateless)** 방식으로 동작해요. 즉, 각 호출이 독립적이기 때문에 이전 대화 내용을 기억하지 못해요. 멀티턴(multi-turn) 대화를 구현하려면 **체크포인터(Checkpointer)** 가 필요해요.

체크포인터는 그래프의 각 실행 단계 후 상태(State)를 자동으로 저장하고, 동일한 `thread_id`로 재호출할 때 저장된 상태를 불러와 대화를 이어갑니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `MemorySaver`를 생성하고 그래프에 체크포인터로 적용할 수 있어요
2. `thread_id`를 사용해 멀티턴 대화 세션을 구분하고 관리할 수 있어요
3. `RunnableConfig`의 `recursion_limit`과 `configurable`을 올바르게 설정할 수 있어요
4. `get_state(config)`로 저장된 스냅샷(values, config, next, metadata)을 조회할 수 있어요
5. `thread_id` 변경으로 독립 세션을 생성하고 차이를 실습할 수 있어요

## 사전 지식

- `06-Tools-Integration.ipynb`: Tool Binding, ToolNode, tools_condition, Conditional Edges
- `05-ChatBot.ipynb`: StateGraph, add_messages 리듀서, 기본 챗봇 구성
- `04-StateGraph-Basics.ipynb`: StateGraph, State, Node, Edge 개념


## 체크포인터 개념과 아키텍처

### 왜 체크포인터가 필요한가요?

LangGraph의 그래프는 기본적으로 상태 비보존 방식이에요. 사용자가 "제 이름은 홍길동이에요"라고 말하고, 다음 호출에서 "제 이름이 뭐죠?"라고 물어보면 답을 모르는 상태가 됩니다.

> **비유: 메모장 없는 상담원 vs 메모장 있는 상담원**
>
> 체크포인터가 없는 그래프는 **메모장 없이 전화 상담하는 직원**과 같아요. 전화를 끊고 다시 걸면 이전에 무슨 이야기를 했는지 전혀 기억 못 해요. 체크포인터를 추가하면 **메모장에 대화 내용을 기록하는 직원**이 돼요. 같은 고객(같은 `thread_id`)이 다시 전화하면 메모장을 펼쳐서 이전 대화를 떠올릴 수 있어요.

체크포인터를 추가하면:
- 각 노드 실행 후 **상태(State)를 자동 저장**해요
- `thread_id`로 **대화 세션을 구분**해요
- 동일한 `thread_id` 재호출 시 **이전 상태를 복원**해요

### 체크포인터 종류

| 체크포인터 | 저장 방식 | 용도 |
|------------|-----------|------|
| `MemorySaver` | 메모리(RAM) | 개발/테스트 환경 |
| `SqliteSaver` | SQLite 파일 | 단일 서버 영구 저장 |
| `PostgresSaver` | PostgreSQL DB | 프로덕션 환경 |
| `RedisSaver` | Redis | 고성능 분산 환경 |

### 전체 아키텍처

```mermaid
flowchart TD
    U(["사용자 입력<br>User Input"]):::input
    C{"configurable<br>thread_id"}:::process
    START(["START"]):::input
    chatbot["chatbot 노드<br>LLM + Tools"]:::process
    tools["tools 노드<br>ToolNode"]:::process
    END_NODE(["END"]):::output
    MEM[("MemorySaver<br>체크포인트 저장")]:::storage

    U --> C
    C -->|"thread_id: 1<br>(세션 유지)"| START
    START --> chatbot
    chatbot -->|"도구 필요?"| tools
    tools --> chatbot
    chatbot --> END_NODE
    chatbot -.->|"상태 저장"| MEM
    tools -.->|"상태 저장"| MEM
    MEM -.->|"상태 복원<br>(같은 thread_id)"| chatbot

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### 핵심 구성 요소

| 구성 요소 | 역할 | 설명 |
|-----------|------|------|
| `MemorySaver` | 체크포인터 | 메모리에 상태 저장/복원 |
| `thread_id` | 세션 식별자 | 대화 세션을 고유하게 구분 |
| `RunnableConfig` | 실행 설정 | thread_id, recursion_limit 전달 |
| `get_state(config)` | 스냅샷 조회 | 저장된 상태 확인 |
| `StateSnapshot` | 상태 스냅샷 | values, config, next, metadata 포함 |


## 환경 설정


In [1]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어요)
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import os

# LangSmith 추적 설정 (선택 사항)
# 실행 흐름을 LangSmith에서 시각적으로 확인할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Part02-Memory"


## 1. MemorySaver 체크포인터 생성

`MemorySaver`는 LangGraph에서 가장 간단한 형태의 체크포인터예요. 상태를 파이썬 프로세스의 메모리에 저장합니다.


In [1]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()


/Users/elixir/dev/lecture/hk-toss/10_langgraph/.venv/lib/python3.14/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. 메모리가 있는 에이전트 그래프 구성

이전 노트북(`06-Tools-Integration.ipynb`)에서 만든 Tool + ToolNode + tools_condition 패턴에 체크포인터를 추가해요. 여기서는 `TavilySearch`를 웹 검색 도구로 사용해요.

### checkpointer 유무 비교

| 구분 | `compile()` | `compile(checkpointer=memory)` |
|------|-------------|-------------------------------|
| 대화 기억 | 매번 새로운 대화 | `thread_id`로 대화 이어가기 |
| 상태 저장 | 실행 후 상태 소멸 | 노드 실행마다 자동 저장 |
| `get_state()` | 사용 불가 | 스냅샷 조회 가능 |
| HITL(사람 개입) | 구현 불가 | `interrupt`로 중단/재개 가능 |


In [2]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=3)
tools = [search_tool]

In [9]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = init_chat_model("openai:gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")


### 체크포인터를 적용하여 그래프 컴파일

핵심은 `compile()` 호출 시 `checkpointer=memory`를 전달하는 것이에요.


In [10]:
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

{'recursion_limit': 10, 'configurable': {'thread_id': 'user_001'}}

## 3. RunnableConfig 설정과 thread_id

`RunnableConfig`는 그래프 실행 시 전달하는 실행 설정이에요. 체크포인터와 함께 사용할 때 두 가지 중요한 설정이 있어요:

| 설정 | 키 | 설명 |
|------|----|------|
| `recursion_limit` | 최상위 키 | 최대 방문 노드 수 (무한 루프 방지) |
| `configurable.thread_id` | `configurable` 내부 | 대화 세션 식별자 |


In [13]:
from langchain_core.runnables import RunnableConfig

config_thread_1 = RunnableConfig(
    recursion_limit=10,
    configurable={"thread_id": "user_001"}
)

config_thread_1

{'recursion_limit': 10, 'configurable': {'thread_id': 'user_001'}}

## 4. 멀티턴 대화 테스트

이제 체크포인터가 실제로 동작하는지 확인해볼게요. 첫 번째 대화에서 이름을 알려주고, 두 번째 대화에서 이름을 물어봐요.


In [14]:
inputs = {
    "messages": [("user", "안녕하세요 저는 엘릭서입니다. 랭그래프를 공부하고있어요.")]
}

# 그래프 실행 결과를 스트리밍으로 출력해요
for chunk in graph.stream(inputs, config_thread_1, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        print(f"\n--- {node_name} 노드 ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                msg.pretty_print()


--- chatbot 노드 ---
================================== Ai Message ==================================

안녕하세요! 엘릭서와 랭그래프를 공부하고 계신다니 멋지네요. 랭그래프에 대해 어떤 주제나 질문이 있으신가요? 도움이 필요하시면 언제든지 말씀해 주세요!


In [ ]:
inputs = {
    "messages": [("user", "엘릭서와 랭그래프를 공부하고 있는게 아니고 랭그래프를 공부한다고. 내 이름이 엘릭서고")]
}

# 그래프 실행 결과를 스트리밍으로 출력해요
for chunk in graph.stream(inputs, config_thread_1, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        print(f"\n--- {node_name} 노드 ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                msg.pretty_print()


--- chatbot 노드 ---
================================== Ai Message ==================================

알겠습니다, 엘릭서님! 랭그래프를 공부하고 계신 거군요. 랭그래프에 대해 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요. 도와드리겠습니다!


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 세 번째 대화: 도구 호출 포함
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 다른 thread_id로 독립 세션 테스트

`thread_id`를 변경하면 완전히 새로운 독립 세션이 시작돼요. 이전 대화 내용을 전혀 모르는 상태가 됩니다.


In [17]:
inputs = {
    "messages": [("user", "내 이름이 뭐라고?")]
}

config_thread_2 = RunnableConfig(
    recursion_limit=10,
    configurable={"thread_id": "user_002"}
)
# 그래프 실행 결과를 스트리밍으로 출력해요
for chunk in graph.stream(inputs, config_thread_2, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        print(f"\n--- {node_name} 노드 ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                msg.pretty_print()


--- chatbot 노드 ---
================================== Ai Message ==================================

당신의 이름은 제가 알 수 없습니다. 이름을 말씀해 주시면 기억하겠습니다!


## 6. 스냅샷(Snapshot)으로 저장 상태 조회

지금까지 체크포인터로 대화를 저장하고 이어가는 방법을 배웠어요. 그런데 저장된 상태를 **직접 열어보고** 싶을 때가 있어요. 디버깅할 때, 또는 사용자에게 대화 기록을 보여줄 때요.

`get_state(config)` 메서드를 사용하면 특정 `thread_id`의 현재 저장 상태를 확인할 수 있어요. 반환되는 `StateSnapshot` 객체에는 다음 정보가 담겨 있어요:

| 속성 | 타입 | 설명 |
|------|------|------|
| `values` | dict | 현재 저장된 State 값 (메시지 목록 등) |
| `config` | dict | checkpoint_id, thread_id 등 설정 정보 |
| `next` | tuple | 다음에 실행될 노드 이름 (완료 시 빈 튜플) |
| `metadata` | dict | step 번호, source, parents 등 실행 메타데이터 |


In [31]:
snapshot = graph.get_state(config_thread_1)
# snapshot

print(snapshot.values["messages"])
print(snapshot.config)
print(snapshot.next)
print(snapshot.metadata)

[HumanMessage(content='안녕하세요 저는 엘릭서입니다. 랭그래프를 공부하고있어요.', additional_kwargs={}, response_metadata={}, id='db84a320-d755-4f1e-9428-28865a6c52ae'), AIMessage(content='안녕하세요! 엘릭서와 랭그래프를 공부하고 계신다니 멋지네요. 랭그래프에 대해 어떤 주제나 질문이 있으신가요? 도움이 필요하시면 언제든지 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 1206, 'total_tokens': 1259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1152}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c96d4b6b5b', 'id': 'chatcmpl-DfborfiiykbZNZBCVQwYwjgNXvnUF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e294a-522d-7122-9eae-8d8171520531-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1206, 'output_tokens': 53, 'total_tokens': 1259, 

In [32]:
# ---------------------------------------------------
# values: 현재 저장된 State 값
# ---------------------------------------------------
# messages 키 아래에 대화 내용 전체가 저장돼요
# ==================================================
# 저장된 메시지 목록 (values['messages'])
# ==================================================

messages = snapshot.values["messages"]
print(f"총 {len(messages)}개의 메시지가 저장됨")
print()

for i, msg in enumerate(messages):
    print(f"[{i+1}] {msg.__class__.__name__}: {str(msg.content)[:80]}...")

총 6개의 메시지가 저장됨

[1] HumanMessage: 안녕하세요 저는 엘릭서입니다. 랭그래프를 공부하고있어요....
[2] AIMessage: 안녕하세요! 엘릭서와 랭그래프를 공부하고 계신다니 멋지네요. 랭그래프에 대해 어떤 주제나 질문이 있으신가요? 도움이 필요하시면 언제든지 말씀해 ...
[3] HumanMessage: 내 이름이 뭐라고?...
[4] AIMessage: 당신의 이름은 "엘릭서"입니다. 도움이 더 필요하시면 말씀해 주세요!...
[5] HumanMessage: 엘릭서와 랭그래프를 공부하고 있는게 아니고 랭그래프를 공부한다고. 내 이름이 엘릭서고...
[6] AIMessage: 알겠습니다, 엘릭서님! 랭그래프를 공부하고 계신 거군요. 랭그래프에 대해 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요. 도와드리겠습니다!...


In [33]:
# ---------------------------------------------------
# 메시지 상세 정보 출력 헬퍼 함수
# ---------------------------------------------------
# 메시지 트리를 직접 출력하는 구현이에요
def print_message_detail(msg, index: int = 0):
    """메시지의 상세 정보를 보기 좋게 출력해요"""
    print(f"\n{'='*40}")
    print(f"[메시지 {index+1}] {msg.__class__.__name__}")
    print(f"{'='*40}")

    # 내용(content) 출력
    content = msg.content
    if isinstance(content, list):
        # Content Block 형식 (Anthropic 등)
        for block in content:
            if isinstance(block, dict) and "text" in block:
                print(f"content: {block['text'][:100]}")
    elif content:
        print(f"content: {str(content)[:100]}")
    else:
        # content: (없음)
        pass

    # 도구 호출 정보 출력
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"tool_calls: {len(msg.tool_calls)}개")
        for tc in msg.tool_calls:
            print(f"  - 도구: {tc['name']}, 인수: {tc['args']}")

    # 도구 호출 ID (ToolMessage)
    if hasattr(msg, "tool_call_id"):
        print(f"tool_call_id: {msg.tool_call_id}")


# 저장된 모든 메시지 상세 출력
# thread_id 'user_001'의 저장된 메시지 상세:
messages = snapshot.values["messages"]
for i, msg in enumerate(messages):
    print_message_detail(msg, i)


[메시지 1] HumanMessage
content: 안녕하세요 저는 엘릭서입니다. 랭그래프를 공부하고있어요.

[메시지 2] AIMessage
content: 안녕하세요! 엘릭서와 랭그래프를 공부하고 계신다니 멋지네요. 랭그래프에 대해 어떤 주제나 질문이 있으신가요? 도움이 필요하시면 언제든지 말씀해 주세요!

[메시지 3] HumanMessage
content: 내 이름이 뭐라고?

[메시지 4] AIMessage
content: 당신의 이름은 "엘릭서"입니다. 도움이 더 필요하시면 말씀해 주세요!

[메시지 5] HumanMessage
content: 엘릭서와 랭그래프를 공부하고 있는게 아니고 랭그래프를 공부한다고. 내 이름이 엘릭서고

[메시지 6] AIMessage
content: 알겠습니다, 엘릭서님! 랭그래프를 공부하고 계신 거군요. 랭그래프에 대해 궁금한 점이나 필요한 정보가 있다면 말씀해 주세요. 도와드리겠습니다!


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: next: 다음 실행될 노드
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: metadata: 실행 메타데이터
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 스냅샷 상세 분석

저장된 각 메시지의 상세 정보를 확인하는 헬퍼 함수를 직접 구현해볼게요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 상세 정보 출력 헬퍼 함수
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 대화 이력 전체 조회

`get_state_history(config)` 메서드를 사용하면 특정 스레드의 **모든 과거 체크포인트**를 순서대로 확인할 수 있어요.


In [36]:
history = list(graph.get_state_history(config_thread_1))

print(f"총 {len(history)}개의 체크포인트가 저장됨\n")

for i, state in enumerate(history):
    # 각 체크포인트의 요약 정보 출력
    checkpoint_id = state.config["configurable"].get("checkpoint_id", "N/A")
    step = state.metadata.get("step", -1)
    msg_count = len(state.values.get("messages", []))
    next_nodes = state.next

    print(f"[체크포인트 {i+1}]")
    print(f"  step: {step}")
    print(f"  저장된 메시지 수: {msg_count}")
    print(f"  다음 노드: {next_nodes if next_nodes else '(완료)'}")
    print(f"  checkpoint_id: {checkpoint_id[:20]}...")
    print()

총 9개의 체크포인트가 저장됨

[체크포인트 1]
  step: 7
  저장된 메시지 수: 6
  다음 노드: (완료)
  checkpoint_id: 1f14fff1-750b-6028-8...

[체크포인트 2]
  step: 6
  저장된 메시지 수: 5
  다음 노드: ('chatbot',)
  checkpoint_id: 1f14fff1-5ebc-6f74-8...

[체크포인트 3]
  step: 5
  저장된 메시지 수: 4
  다음 노드: ('__start__',)
  checkpoint_id: 1f14fff1-5ebb-6a20-8...

[체크포인트 4]
  step: 4
  저장된 메시지 수: 4
  다음 노드: (완료)
  checkpoint_id: 1f14fff0-8928-61ec-8...

[체크포인트 5]
  step: 3
  저장된 메시지 수: 3
  다음 노드: ('chatbot',)
  checkpoint_id: 1f14fff0-7c26-6336-8...

[체크포인트 6]
  step: 2
  저장된 메시지 수: 2
  다음 노드: ('__start__',)
  checkpoint_id: 1f14fff0-7c24-6d4c-8...

[체크포인트 7]
  step: 1
  저장된 메시지 수: 2
  다음 노드: (완료)
  checkpoint_id: 1f14ffef-bca8-64be-8...

[체크포인트 8]
  step: 0
  저장된 메시지 수: 1
  다음 노드: ('chatbot',)
  checkpoint_id: 1f14ffef-aab4-6082-8...

[체크포인트 9]
  step: -1
  저장된 메시지 수: 0
  다음 노드: ('__start__',)
  checkpoint_id: 1f14ffef-aab1-6d00-b...



## 실습: 여러 사용자 시뮬레이션

아래 TODO 블록을 완성해 보세요. 두 명의 다른 사용자가 각자 대화하고, 서로의 정보를 기억하지 못하는 것을 확인해요.


In [1]:
import uuid
from langchain_core.runnables import RunnableConfig

# ============================================================
# 구현 예시: 두 명의 사용자 대화 시뮬레이션
# ============================================================
thread_a = str(uuid.uuid4())
thread_b = str(uuid.uuid4())

print(f"사용자 A thread_id: {thread_a}")
print(f"사용자 B thread_id: {thread_b}")

config_a = RunnableConfig(recursion_limit=10, configurable={"thread_id": thread_a})
config_b = RunnableConfig(recursion_limit=10, configurable={"thread_id": thread_b})


def run_user_turn(label: str, config: RunnableConfig, message: str) -> None:
    """한 사용자 턴을 실행하고 마지막 메시지를 출력해요."""
    print(f"\n[{label}] 사용자: {message}")
    for chunk in graph.stream({"messages": [("user", message)]}, config, stream_mode="updates"):
        for node_name, node_output in chunk.items():
            if "messages" in node_output:
                last_message = node_output["messages"][-1]
                print(f"--- {label} / {node_name} ---")
                last_message.pretty_print()


run_user_turn("사용자 A", config_a, "저는 이순신 장군의 팬이에요")
run_user_turn("사용자 B", config_b, "저는 세종대왕의 팬이에요")
run_user_turn("사용자 A", config_a, "제가 누구의 팬이라고 했죠?")
run_user_turn("사용자 B", config_b, "제가 누구의 팬이라고 했죠?")

사용자 A thread_id: cfcc4985-a380-443d-97a2-ce865febea9f
사용자 B thread_id: 39150e47-d7ce-4221-a938-08cddd19ad54

[사용자 A] 사용자: 저는 이순신 장군의 팬이에요


NameError: name 'graph' is not defined

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`MemorySaver`**: 메모리 기반 인메모리 체크포인터로, 개발/테스트 환경에서 빠르게 상태 저장을 테스트할 수 있어요. 프로덕션에서는 영구 저장소로 교체해야 해요.

- **`checkpointer=memory`**: `compile()` 시 체크포인터를 전달하면, 이후 모든 노드 실행 후 State가 자동으로 저장돼요. 그래프 구성 자체는 변경이 없어요.

- **`thread_id`**: `RunnableConfig`의 `configurable.thread_id`로 대화 세션을 구분해요. 같은 ID = 연속 대화, 다른 ID = 독립 세션이에요.

- **`RunnableConfig`**: `recursion_limit`으로 무한 루프를 방지하고, `configurable`에 `thread_id`를 담아 그래프에 전달해요.

- **`get_state(config)`**: 특정 스레드의 현재 저장 상태를 조회해요. `values`, `config`, `next`, `metadata` 속성으로 상태 전체를 확인할 수 있어요.

- **`get_state_history(config)`**: 특정 스레드의 모든 과거 체크포인트를 역순으로 조회해요. Time Travel의 기반이 돼요.


## 다음 노트북 예고

다음 `08-Human-In-The-Loop.ipynb`에서는 **Human-in-the-Loop(사람 개입)** 패턴을 배워요. 체크포인터가 제공하는 상태 저장/복원 기능을 활용해, 그래프 실행 중간에 사람이 개입하고 승인/수정할 수 있는 `interrupt`와 `Command` 패턴을 다뤄요. 또한 과거 특정 체크포인트로 돌아가는 **Time Travel** 기능도 살펴볼게요!
